<a href="https://colab.research.google.com/github/inhyuk78/Re-implementation-of-MOLI-By-Hossein-Sharifi-Noghabi-/blob/main/01_data_import_and_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path_drug_response = '/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/GDSC2_fitted_dose_response_27Oct23.xlsx'
path_gene_expr = '/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Gene_Expression/rnaseq_merged.csv'
path_mutation = '/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Mutation/mutations_all_20260316.csv'
path_CNA = '/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Copy_Number_Aberration/CNV_genes_20250207.csv'

In [ ]:
import pandas as pd
import numpy as np

# **Importing Dataset**

### (i) GDSC2 Drug Response Dataset

In [ ]:
drug_df = pd.read_excel(path_drug_response)

In [ ]:
cols = drug_df.columns
print(cols)

Index(['DATASET', 'NLME_RESULT_ID', 'NLME_CURVE_ID', 'CELL_LINE_NAME',
       'SANGER_MODEL_ID', 'CANCER_TYPE', 'DRUG_ID', 'DRUG_NAME',
       'PUTATIVE_TARGET', 'PATHWAY_NAME', 'MIN_CONC', 'MAX_CONC', 'LN_IC50',
       'AUC', 'RMSE', 'Z_SCORE'],
      dtype='object')


In [ ]:
drug_df_cols = [
    'CELL_LINE_NAME',
    'SANGER_MODEL_ID', # Unique identifier of each cell line
    'CANCER_TYPE',
    'DRUG_ID',
    'DRUG_NAME',
    'PUTATIVE_TARGET',
    'LN_IC50', # Log of concen required to reduce cell viability by 50% (lower = less drug required to reduce)
                ]

drug_df = drug_df[drug_df_cols]
drug_df.head()

,CELL_LINE_NAME,SANGER_MODEL_ID,CANCER_TYPE,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,LN_IC50
0,PFSK-1,SIDM01132,Other Solid Cancers,1003,Camptothecin,TOP1,-1.463887
1,A673,SIDM00848,Ewing's Sarcoma,1003,Camptothecin,TOP1,-4.869455
2,ES5,SIDM00263,Ewing's Sarcoma,1003,Camptothecin,TOP1,-3.360586
3,ES7,SIDM00269,Ewing's Sarcoma,1003,Camptothecin,TOP1,-5.044940
4,EW-11,SIDM00203,Ewing's Sarcoma,1003,Camptothecin,TOP1,-3.741991


In [ ]:
drug_ids = list(drug_df['SANGER_MODEL_ID'].unique())
print(len(drug_ids))
print(drug_ids)

969
['SIDM01132', 'SIDM00848', 'SIDM00263', 'SIDM00269', 'SIDM00203', 'SIDM01111', 'SIDM00909', 'SIDM00807', 'SIDM01085', 'SIDM01160', 'SIDM01190', 'SIDM00889', 'SIDM00905', 'SIDM00627', 'SIDM00982', 'SIDM00998', 'SIDM00799', 'SIDM00581', 'SIDM01171', 'SIDM01193', 'SIDM01189', 'SIDM00315', 'SIDM00924', 'SIDM01120', 'SIDM00922', 'SIDM00512', 'SIDM00548', 'SIDM00734', 'SIDM00747', 'SIDM00746', 'SIDM00745', 'SIDM00742', 'SIDM00737', 'SIDM00769', 'SIDM00709', 'SIDM00716', 'SIDM00729', 'SIDM00727', 'SIDM00726', 'SIDM00724', 'SIDM01122', 'SIDM00509', 'SIDM00865', 'SIDM00522', 'SIDM00654', 'SIDM00653', 'SIDM00741', 'SIDM00760', 'SIDM00735', 'SIDM00706', 'SIDM00699', 'SIDM00733', 'SIDM00730', 'SIDM00719', 'SIDM00965', 'SIDM01128', 'SIDM01126', 'SIDM01121', 'SIDM01131', 'SIDM00713', 'SIDM01100', 'SIDM01099', 'SIDM01179', 'SIDM00320', 'SIDM00293', 'SIDM00368', 'SIDM00365', 'SIDM00294', 'SIDM01098', 'SIDM00715', 'SIDM00702', 'SIDM00739', 'SIDM01124', 'SIDM01184', 'SIDM01148', 'SIDM01101', 'SIDM00

### (ii) Gene Expression Dataset

In [ ]:
chunks = []

gene_expr_cols = ['model_name', 'model_id', 'gene_symbol', 'rsem_tpm']

for chunk in pd.read_csv(path_gene_expr,chunksize=500_000):
  filtered = chunk[chunk['model_id'].isin(drug_ids)][gene_expr_cols]

  chunks.append(filtered)

rnaseq_df = pd.concat(chunks, ignore_index=True)
rnaseq_df.head()

,model_name,model_id,gene_symbol,rsem_tpm
0,LS-123,SIDM00776,DVL2,3.5521
1,MOLM-13,SIDM00437,STPG1,2.2898
2,HARA,SIDM00598,STPG1,2.9486
3,HCC-366,SIDM01070,STPG1,2.4906
4,MV-4-11,SIDM00657,STPG1,1.3674


In [ ]:
print(rnaseq_df.shape)

(36392676, 4)


In [ ]:
missing = rnaseq_df.isna().mean().sort_values(ascending=False)
missing

,0
rsem_tpm,0.003713
model_name,0.000000
model_id,0.000000
gene_symbol,0.000000


In [ ]:
print(f"Before: {rnaseq_df.shape}")

rnaseq_df = rnaseq_df.dropna()

print(f"After: {rnaseq_df.shape}")

Before: (36392676, 4)
After: (36257567, 4)


### (iii) Mutation Dataset

In [ ]:
chunks = []

mutation_cols = ['model_name', 'model_id', 'gene_symbol', 'cancer_driver', 'effect']

for chunk in pd.read_csv(path_mutation, chunksize=500_00):
  filtered = chunk[chunk['model_id'].isin(drug_ids)][mutation_cols]

  chunks.append(filtered)

mutation_df = pd.concat(chunks, ignore_index=True)
mutation_df.head()

/tmp/ipykernel_756/3596565142.py:5: DtypeWarning: Columns (2,8,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path_mutation, chunksize=500_00):


,model_name,model_id,gene_symbol,cancer_driver,effect
0,GR-ST,SIDM01259,TEKT4,f,ess_splice
1,GR-ST,SIDM01259,EIF5B,f,frameshift
2,GR-ST,SIDM01259,NPHP1,f,intronic
3,GR-ST,SIDM01259,NPHP1,f,frameshift
4,GR-ST,SIDM01259,MIR1302-3,f,nc_variant


In [ ]:
mutation_df['cancer_driver'] = mutation_df['cancer_driver'].map({'t':1, 'f':0})
mutation_df['cancer_driver'].unique()

array([0, 1])

### (iv) Copy Number Aberration (CNA) Dataset

In [ ]:
chunks = []

CNA_cols = ['model_name', 'model_id', 'symbol', 'total_copy_number', 'cn_category', 'loh', 'focal', 'cancer_driver']

CNA_dtype_dict = {
    'model_id': str,
    'model_name': str,
    'cn_category': str,
    'symbol': str,
    'total_copy_number': 'float64',
    'loh': str,
    'focal': str,
    'cancer_driver': int
}

for chunk in pd.read_csv(path_CNA, chunksize=500_000, dtype=CNA_dtype_dict, low_memory=False):
  filtered = chunk[chunk['model_id'].isin(drug_ids)][CNA_cols]

  chunks.append(filtered)

CNA_df = pd.concat(chunks, ignore_index=True)

In [ ]:
CNA_df['loh'] = CNA_df['loh'].map({'t':1, 'f':0})
CNA_df['focal'] = CNA_df['focal'].map({'t':1, 'f':0})

In [ ]:
print(CNA_df['loh'].unique())
print(CNA_df['focal'].unique())

[nan  0.  1.]
[nan  0.  1.]


In [ ]:
missing = CNA_df.isna().mean().sort_values(ascending=False)
missing

# Roughly 2~3% of data NaN values for loh, focal, total_copy_number

,0
loh,0.027934
total_copy_number,0.026916
focal,0.026916
model_name,0.000000
symbol,0.000000
model_id,0.000000
cn_category,0.000000
cancer_driver,0.000000


In [ ]:
print(f"Before: {CNA_df.shape}")

CNA_df = CNA_df.dropna()

print(f"After: {CNA_df.shape}")

Before: (19203438, 8)
After: (18667013, 8)


### Saving pandas DataFrames

In [ ]:
drug_df.to_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Dataframes/drug.parquet')
rnaseq_df.to_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Dataframes/rnaseq.parquet')
mutation_df.to_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Dataframes/mutation.parquet')
CNA_df.to_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Dataframes/CNA.parquet')